# Cascade hybrid recommender

Implementation of a two-stage candidate generation + reranking pipeline

Architecture:
1. Item-Item CF (Cosine Similarity) retrieves top 100 relevant candidates based on crowd behavior. High recall
2. Content-Based (TF-IDF on genres/titles) sorts ONLY these 100 top candidates to match the user's semantic preferences


Who will benefit:
- users with specific niche thematic tastes, eg. strong preference for Sci-Fi inside a broader set of CF recommendations
- tail items that match the user's content profile but lack enough ratings to organically reach the top 10 in pure CF

In [6]:
import sys
from pathlib import Path
import pandas as pd
import numpy as np
from tqdm.auto import tqdm

# adding src to path
current_dir = Path.cwd()
project_root = current_dir.parent
if str(project_root) not in sys.path:
    sys.path.append(str(project_root))

from src.data import loader, splitter
from src.evaluation import metrics
from src.models.content_based import ContentBasedRecommender
from src.models.collaborative import ItemItemRecommender
from src.models.hybrid import CascadeHybridRecommender


## Setup
Using the team's data loader and temporal splitter


In [7]:
movies = loader.load_movies()
ratings = loader.load_ratings()

train_df, val_df, test_df = splitter.split_temporal(
    dataframe=ratings,
    train_ratio=0.7,
    validation_ratio=0.15,
    test_ratio=0.15
)

full_test_df = pd.concat([val_df, test_df])

train_pivot = train_df.pivot(
    index='user_id',
    columns='item_id',
    values='rating'
).fillna(0)

print(f"Train pivot shape: {train_pivot.shape}")


Loaded 3883 movies from /Users/markmatviyiv/studyspace/Recommender Systems/ucu-recsys/data/ml-1m/movies.dat
Loaded 1000209 ratings from /Users/markmatviyiv/studyspace/Recommender Systems/ucu-recsys/data/ml-1m/ratings.dat
Train pivot shape: (6040, 3649)


## Model training
Training base CF and CB models, then wrapping them in the Cascade Hybrid

In [8]:
cb_tfidf = ContentBasedRecommender(similarity_method='tfidf')
cb_tfidf.fit(movies)

cf_cosine = ItemItemRecommender(method='cosine')
cf_cosine.fit(train_df)

hybrid_model = CascadeHybridRecommender(generator_model=cf_cosine, reranker_model=cb_tfidf, num_candidates=100)
hybrid_model.fit(train_pivot)


## Evaluation
Helper functions to compute ranking metrics for K=10, 20, 50

In [9]:
all_results = []

def evaluate_model(model, model_name, test_df, train_pivot, n_users=200, k_list=[10, 20, 50]):
    test_users = test_df['user_id'].unique()[:n_users]
    
    # Filter test data to only include relevant (positive) items for ranking evaluation
    test_positive = test_df[test_df['rating'] >= 4]
    subset_test = test_positive[test_positive['user_id'].isin(test_users)]
    
    recs_list = []
    max_k = max(k_list)
    
    is_cb = isinstance(model, ContentBasedRecommender)
    
    for user_id in tqdm(test_users, desc=f"Evaluating {model_name}", leave=False):
        if is_cb:
            user_recs = model.recommend(user_id, train_pivot, top_k=max_k)
        else:
            user_recs = model.recommend(user_id, top_k=max_k)
            
        if not user_recs.empty:
            recs_list.append(user_recs)
            
    if not recs_list:
        return
        
    all_recs = pd.concat(recs_list)
    
    for k in k_list:
        metrics_res = metrics.compute_ranking_metrics(subset_test, all_recs, top_k=k)
        metrics_res['Model'] = model_name
        metrics_res['K'] = k
        all_results.append(metrics_res)
        
    return


def display_results(results_list):
    if not results_list: return
    df = pd.DataFrame(results_list)
    
    k_values = sorted(df['K'].unique())
    
    for k in k_values:
        print(f"\n--- Metrics @ K={k} ---")
        subset = df[df['K'] == k].drop(columns=['K']).set_index('Model')
        cols = ['ndcg', 'map', 'precision', 'recall']
        subset = subset[cols]
        display(subset.round(4))


## Run
Evaluating baseline models vs the Cascade pipeline on a subset of 200 users

In [10]:
evaluate_model(cf_cosine, "CF (Cosine)", full_test_df, train_pivot, n_users=200)

evaluate_model(cb_tfidf, "CB (TF-IDF)", full_test_df, train_pivot, n_users=200)

evaluate_model(hybrid_model, "Cascade hybrid (CF -> CB)", full_test_df, train_pivot, n_users=200)

display_results(all_results)



--- Metrics @ K=10 ---


,ndcg,map,precision,recall
Model,,,,
CF (Cosine),0.1670,0.0863,0.1520,0.0928
CB (TF-IDF),0.0236,0.0096,0.0195,0.0132
Cascade hybrid (CF -> CB),0.0891,0.0384,0.0855,0.0530



--- Metrics @ K=20 ---


,ndcg,map,precision,recall
Model,,,,
CF (Cosine),0.1712,0.0738,0.1265,0.1419
CB (TF-IDF),0.0278,0.0086,0.0220,0.0233
Cascade hybrid (CF -> CB),0.0977,0.0354,0.0798,0.0872



--- Metrics @ K=50 ---


,ndcg,map,precision,recall
Model,,,,
CF (Cosine),0.1974,0.0711,0.0929,0.2531
CB (TF-IDF),0.0439,0.0102,0.0233,0.0672
Cascade hybrid (CF -> CB),0.1406,0.0407,0.0775,0.2023


## 5. Final conclusions

Some thoughts after running these experiments:

1. CF is still the king for dense datasets:\
   The cascade hybrid (NDCG@10 ~0.089) performs worse than pure CF (NDCG@10 ~0.167). Forcing the model to rerank the top 100 CF candidates strictly by genre similarity hurts the ranking for users with rich histories

2. Massive improvement over pure Content-Based:\
   The hybrid completely destroys pure CB filtering (NDCG 0.089 vs 0.023). Filtering content signals through a CF generator first is incredibly effective compared to global content matching

3. When to use this pipeline:\
   In a real production system, this exact setup shouldn't be used for all active users. It acts as an excellent fallback for the cold-start scenario (users with sparse histories where CF is noisy) or for highly specific contextual queries